# Baseline Model Results

Fits and evaluates the two baseline models (the team-independent historical frequency model and the non-Bayesian team-strength Poisson model).

We expect the Bayesian model to out-perform these two baseline models.

## Imports

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

In [2]:
# Find root directory of project
root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "config.py").exists())
sys.path.insert(0, str(root))

In [3]:
from src.config import PROCESSED_DATA_DIR, TABLES_DIR, TRAIN_SEASONS, TEST_SEASONS
from src.baseline.models import HistoricalFrequencyBaseline, NonBayesianPoissonBaseline

## Evaluation

### Load Training and Testing Data

In [4]:
df = pd.read_csv(PROCESSED_DATA_DIR / "epl_matches.csv", parse_dates = ["Date"])

train_df = df[df["Season"].isin(TRAIN_SEASONS)].reset_index(drop = True)
test_df = df[df["Season"].isin(TEST_SEASONS)].reset_index(drop = True)

print(f"Train: {len(train_df)} matches")
print(f"Test: {len(test_df)} matches")

Train: 3040 matches
Test: 760 matches


### Historical Frequency Model

In [5]:
freq_model = HistoricalFrequencyBaseline().fit(train_df)
freq_preds = freq_model.predict(test_df)

In [6]:
freq_preds[["HomeTeam", "AwayTeam", "Result", "ProbHomeWin", "ProbDraw", "ProbAwayWin"]].head()

,HomeTeam,AwayTeam,Result,ProbHomeWin,ProbDraw,ProbAwayWin
0,Man United,Fulham,H,0.453618,0.225658,0.320724
1,West Ham,Aston Villa,A,0.453618,0.225658,0.320724
2,Nott'm Forest,Bournemouth,D,0.453618,0.225658,0.320724
3,Newcastle,Southampton,H,0.453618,0.225658,0.320724
4,Everton,Brighton,A,0.453618,0.225658,0.320724


In [7]:
print("Historical Frequency Baseline Model")
print(f"Estimated Home Win Percentage: {freq_model.prob_home_win:.3f}")
print(f"Estimated Draw Percentage: {freq_model.prob_draw:.3f}")
print(f"Estimated Away Win Percentage: {freq_model.prob_away_win:.3f}")

Historical Frequency Baseline Model
Estimated Home Win Percentage: 0.454
Estimated Draw Percentage: 0.226
Estimated Away Win Percentage: 0.321


### Non-Bayesian Poisson Model

In [8]:
non_bayesian_poisson_model = NonBayesianPoissonBaseline().fit(train_df)
non_bayesian_poisson_preds = non_bayesian_poisson_model.predict(test_df)

In [9]:
non_bayesian_poisson_preds[["HomeTeam", "AwayTeam", "Result", "ProbHomeWin", "ProbDraw", "ProbAwayWin"]].head()

,HomeTeam,AwayTeam,Result,ProbHomeWin,ProbDraw,ProbAwayWin
0,Man United,Fulham,H,0.650797,0.206261,0.142942
1,West Ham,Aston Villa,A,0.382998,0.255535,0.361467
2,Nott'm Forest,Bournemouth,D,0.513061,0.228081,0.258858
3,Newcastle,Southampton,H,0.542095,0.236038,0.221866
4,Everton,Brighton,A,0.444024,0.274453,0.281523


In [10]:
for model, preds in [("Historical Frequency", freq_preds), ("Non-Bayesian Poisson", non_bayesian_poisson_preds)]:
    pred_labels = preds[["ProbHomeWin", "ProbDraw", "ProbAwayWin"]].idxmax(axis = 1).map({"ProbHomeWin": "H",
                                                                                          "ProbDraw": "D",
                                                                                          "ProbAwayWin": "A"})
    accuracy = (pred_labels == preds["Result"]).mean()
    print(f"{model} Baseline Model Accuracy: {accuracy:.3f}")

Historical Frequency Baseline Model Accuracy: 0.417
Non-Bayesian Poisson Baseline Model Accuracy: 0.441


### Save Evaluation Results

In [11]:
baseline_model_table_output_directory = (TABLES_DIR / "baseline")
baseline_model_table_output_directory.mkdir(parents = True, exist_ok = True)

freq_preds.to_csv(baseline_model_table_output_directory / "historical_frequency_predictions.csv", index = False)
non_bayesian_poisson_preds.to_csv(baseline_model_table_output_directory / "non_bayesian_poisson_predictions.csv", index = False)